# AR Step 1 Latent Quality

Minimal validation notebook for the autoregressive Transformer-VAE from `transformer_vae/train-AR.ipynb`.
It writes reconstruction, prior-generation, interpolation, and functional-family retention artifacts under `model_validation/`.


In [ ]:
from pathlib import Path
import json
import sys
import warnings
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

def find_project_root(start=None):
    p = (Path(start) if start is not None else Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise RuntimeError(f"Couldn't find project root from {p}")

PROJECT_ROOT = "<home>ser/workspace/Smiles-latent-project"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from artifacts.model_compare.ar_model_h256_l256.patched_notebooks import ar_common as arc

ARTIFACT_ROOT = arc.ARTIFACT_ROOT
print("artifact_root =", ARTIFACT_ROOT)
for warning_text in arc.dependency_report():
    warnings.warn(warning_text)


In [ ]:
OUTPUT_DIR = ARTIFACT_ROOT / "model_validation"
TABLES_DIR = OUTPUT_DIR / "tables"
PLOTS_DIR = OUTPUT_DIR / "plots"
FAMILY_PLOTS_DIR = PLOTS_DIR / "phase5_family_traversals"
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR, FAMILY_PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

metrics = {}
warnings_out = []


## Load Dataset And AR Checkpoint


In [ ]:
bundle = arc.load_dataset_and_splits()
model, device, ckpt_path = arc.load_ar_model(bundle)
print("device =", device)
print("checkpoint =", ckpt_path)
print("dataset rows =", len(bundle["df"]))
print("split sizes =", bundle["split_sizes"])
print("vocab size =", bundle["vocab_size"], "max_len =", bundle["max_len"])


## A. Reconstruction Evaluation


In [ ]:
split_arrays = {
    "train": bundle["train_data"],
    "val": bundle["val_data"],
    "test": bundle["test_data"],
}
recon_records, recon_summary = arc.evaluate_reconstruction(
    model,
    split_arrays,
    bundle["id2tok"],
    device,
    batch_size=256,
)
recon_records.to_csv(TABLES_DIR / "reconstruction_samples.csv", index=False)
recon_summary.to_csv(TABLES_DIR / "reconstruction_summary_by_split.csv", index=False)

for row in recon_summary.to_dict("records"):
    split = row["split"]
    for key, value in row.items():
        if key != "split":
            metrics[f"reconstruction/{split}/{key}"] = value

metrics_df = pd.DataFrame([{"metric": k, "value": v} for k, v in metrics.items()])
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
arc.write_json(OUTPUT_DIR / "metrics.json", metrics)
display(recon_summary)


## B. Prior Latent Generation


In [ ]:
N_PRIOR_SAMPLES = 5000

train_smiles = bundle["df"].iloc[bundle["train_idx"]]["smiles"].tolist()
train_canonical = set(filter(None, (arc.canonicalize_smiles(smi) for smi in tqdm(train_smiles, desc="train canonical"))))

prior_samples, prior_summary = arc.sample_prior(
    model,
    n_samples=N_PRIOR_SAMPLES,
    latent_size=arc.LATENT_SIZE,
    id2tok=bundle["id2tok"],
    device=device,
    train_canonical=train_canonical,
    batch_size=256,
)
prior_samples.to_csv(TABLES_DIR / "prior_random_latent_samples.csv", index=False)
pd.DataFrame([prior_summary]).to_csv(TABLES_DIR / "prior_random_latent_summary.csv", index=False)

for key, value in prior_summary.items():
    metrics[f"prior/{key}"] = value
arc.write_json(OUTPUT_DIR / "metrics.json", metrics)
display(pd.DataFrame([prior_summary]))


## C. Latent Interpolation


In [ ]:
torch = arc.torch
N_INTERP_PAIRS = 20
INTERP_STEPS = np.linspace(0.0, 1.0, 11)
rng = np.random.default_rng(arc.SEED)
test_positions = rng.choice(len(bundle["test_data"]), size=(N_INTERP_PAIRS, 2), replace=False)

rows = []
model.eval()
for pair_id, (pos_a, pos_b) in enumerate(tqdm(test_positions, desc="interpolation pairs")):
    x_pair = torch.as_tensor(
        np.stack([bundle["test_data"][pos_a], bundle["test_data"][pos_b]]),
        dtype=torch.long,
        device=device,
    )
    with torch.no_grad():
        mu, _ = model.encode(x_pair)
    z_a, z_b = mu.detach().cpu().numpy()
    z_path = np.stack([(1.0 - t) * z_a + t * z_b for t in INTERP_STEPS]).astype(np.float32)
    decoded = arc.decode_latents(model, z_path, bundle["id2tok"], device, batch_size=128)
    for step, (t, rec) in enumerate(zip(INTERP_STEPS, decoded)):
        rows.append({
            "pair_id": pair_id,
            "step": step,
            "t": float(t),
            "test_pos_a": int(pos_a),
            "test_pos_b": int(pos_b),
            **rec,
        })

interp_df = pd.DataFrame(rows)
interp_df.to_csv(TABLES_DIR / "latent_interpolation_table.csv", index=False)
interp_summary = interp_df.groupby("step", as_index=False)["valid"].mean().rename(columns={"valid": "valid_fraction"})
interp_summary.to_csv(TABLES_DIR / "latent_interpolation_validity_by_step.csv", index=False)

plt.figure(figsize=(6, 4))
plt.plot(interp_summary["step"], interp_summary["valid_fraction"], marker="o")
plt.ylim(-0.05, 1.05)
plt.xlabel("interpolation step")
plt.ylabel("valid fraction")
plt.title("AR latent interpolation validity")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "interp_validity_by_step.png", dpi=180)
plt.show()
display(interp_summary)


## D. Functional-Family Interpolation And Retention


In [ ]:
PHASE5_MIN_GROUP_SIZE = 20
PHASE5_T_VALUES = np.linspace(0.0, 1.0, 11)
patterns = arc.compile_functional_group_patterns()
family_names = [name for name, _ in arc.FUNCTIONAL_GROUP_SMARTS]

test_meta = bundle["df"].iloc[bundle["test_idx"]].reset_index(drop=True).copy()
test_meta["test_pos"] = np.arange(len(test_meta), dtype=int)
test_meta["families"] = [arc.detect_families(smi, patterns=patterns) for smi in tqdm(test_meta["smiles"], desc="test families")]
membership = test_meta.explode("families").dropna(subset=["families"]).rename(columns={"families": "target_family"})
membership.to_csv(TABLES_DIR / "phase5_functional_group_membership.csv", index=False)

available_counts = membership["target_family"].value_counts()
selected_families = [fam for fam in family_names if available_counts.get(fam, 0) >= PHASE5_MIN_GROUP_SIZE]
rng = np.random.default_rng(arc.SEED)
family_rows = []

for pair_id, target_family in enumerate(selected_families):
    family_df = membership[membership["target_family"] == target_family].drop_duplicates("test_pos")
    chosen = rng.choice(family_df["test_pos"].to_numpy(), size=2, replace=False)
    x_pair = arc.torch.as_tensor(bundle["test_data"][chosen], dtype=arc.torch.long, device=device)
    with arc.torch.no_grad():
        mu, _ = model.encode(x_pair)
    z_a, z_b = mu.detach().cpu().numpy()
    z_path = np.stack([(1.0 - t) * z_a + t * z_b for t in PHASE5_T_VALUES]).astype(np.float32)
    decoded = arc.decode_latents(model, z_path, bundle["id2tok"], device, batch_size=128)

    for step, (t, rec) in enumerate(zip(PHASE5_T_VALUES, decoded)):
        matched = arc.detect_families(rec["canonical_smiles"], patterns=patterns) if rec["valid"] else []
        family_rows.append({
            "pair_id": pair_id,
            "target_family": target_family,
            "step": step,
            "t": float(t),
            "test_pos_a": int(chosen[0]),
            "test_pos_b": int(chosen[1]),
            "matched_families": ";".join(matched),
            "retention": int(target_family in matched),
            **rec,
        })

family_df = pd.DataFrame(family_rows)
family_df.to_csv(TABLES_DIR / "phase5_family_traversal_records.csv", index=False)

if len(family_df):
    family_summary = (
        family_df.groupby("target_family", as_index=False)
        .agg(valid_fraction=("valid", "mean"), target_family_retention=("retention", "mean"), n_steps=("step", "count"))
    )
    family_summary.to_csv(TABLES_DIR / "phase5_family_retention_summary.csv", index=False)

    plt.figure(figsize=(9, 4))
    plt.bar(family_summary["target_family"], family_summary["target_family_retention"])
    plt.xticks(rotation=45, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("retention")
    plt.title("Functional-family retention along interpolation paths")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "phase5_family_retention.png", dpi=180)
    plt.show()

    metrics["family_retention/overall_valid_fraction"] = float(family_df["valid"].mean())
    metrics["family_retention/overall_target_family_retention"] = float(family_df["retention"].mean())
    metrics["family_retention/pair_counts_by_target_family"] = {fam: 1 for fam in selected_families}
    arc.write_json(OUTPUT_DIR / "metrics.json", metrics)
    display(family_summary)
else:
    warnings.warn("No eligible functional families were found for interpolation retention.")


In [ ]:
# Average retention across functional families (macro-average)
family_summary_local = globals().get("family_summary")

if family_summary_local is None:
    if "TABLES_DIR" in globals():
        summary_path = TABLES_DIR / "phase5_family_retention_summary.csv"
        family_summary_local = pd.read_csv(summary_path) if summary_path.exists() else None
    else:
        family_summary_local = None

if family_summary_local is not None and len(family_summary_local):
    macro_avg_retention = float(family_summary_local["target_family_retention"].mean())
    print("Average family retention (macro):", macro_avg_retention)
else:
    warnings.warn("family_summary not available; run the Phase 5 family-retention cell first.")
